# Exam: Time Series Visualization with Bokeh

This exam tests your ability to visualize time series data using the Bokeh library.
You will be working with the "Daily Minimum Temperatures in Melbourne" dataset.
For each question, provide the Python code using Bokeh to generate the requested visualization.

**Dataset:** "daily-minimum-temperatures-in-melbourne.csv"

```python
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    DatetimeTickFormatter,
    NumeralTickFormatter,
)
from bokeh.layouts import row, column
from bokeh.transform import factor_cmap

output_notebook()  # Enable Bokeh output in Jupyter Notebook

# Load the Dataset
df = pd.read_csv("daily-minimum-temperatures-in-melbourne.csv")

# Rename columns for clarity
df.columns = ['Date', 'Temperature']

# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Remove '?' from the 'Temperature' column and convert to numeric
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'])

Question 1: Basic Time Series Line Plot
1.  Create a basic line plot showing the daily minimum temperature over time.

    * Use the 'Date' column on the x-axis and the 'Temperature' column on the y-axis.
    * Set the plot title to "Daily Minimum Temperatures".
    * Label the x-axis as "Date" and the y-axis as "Temperature (°C)".
    * Add tooltips to display the date and temperature when hovering over the line.
    * Enable pan, wheel zoom, and reset tools.


In [ ]:
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool, DatetimeTickFormatter

output_notebook()

# Chargement des donnees
df = pd.read_csv("./datasets/daily-minimum-temperatures-in-melbourne.csv")
df.columns = ['Date', 'Temperature']
df['Date'] = pd.to_datetime(df['Date'])
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'])

source = ColumnDataSource(df)

p = figure(title="Daily Minimum Temperatures", x_axis_type="datetime",
           width=900, height=400,
           x_axis_label="Date", y_axis_label="Temperature (°C)",
           tools="pan,wheel_zoom,reset")

p.line('Date', 'Temperature', source=source, line_width=1, color="steelblue")

hover = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Temp", "@Temperature{0.1} °C")],
                  formatters={"@Date": "datetime"})
p.add_tools(hover)

show(p)

Question 2: Rolling Average
2.  Calculate the 30-day rolling average of the daily minimum temperature and plot it
    alongside the original temperature data.

    * Create a new column 'Rolling_Avg' in the DataFrame containing the 30-day rolling average.
    * Plot both the original 'Temperature' and the 'Rolling_Avg' on the same plot.
    * Use different colors and line styles to distinguish between the two.
    * Add a legend to the plot to label the lines.
    * Add tooltips to display the date, original temperature, and rolling average.

In [ ]:
# Moyenne glissante sur 30 jours
df['Rolling_Avg'] = df['Temperature'].rolling(window=30).mean()

source2 = ColumnDataSource(df)

p2 = figure(title="Températures et moyenne glissante 30j", x_axis_type="datetime",
            width=900, height=400,
            x_axis_label="Date", y_axis_label="Temperature (°C)",
            tools="pan,wheel_zoom,reset")

p2.line('Date', 'Temperature', source=source2, line_width=1, color="lightgray",
        legend_label="Température journalière", alpha=0.6)
p2.line('Date', 'Rolling_Avg', source=source2, line_width=2, color="tomato",
        legend_label="Moyenne glissante 30j", line_dash="dashed")

p2.legend.location = "top_left"

hover2 = HoverTool(tooltips=[("Date", "@Date{%F}"),
                              ("Temp", "@Temperature{0.1} °C"),
                              ("Moy. 30j", "@Rolling_Avg{0.1} °C")],
                   formatters={"@Date": "datetime"})
p2.add_tools(hover2)

show(p2)

Question 3: Monthly Box Plots
3.  Create box plots to visualize the distribution of temperatures for each month.

    * Extract the month from the 'Date' column and create a new 'Month' column.
    * Group the data by 'Month' and prepare it for plotting.
    * Use Bokeh's box plot elements to visualize the distribution.
    * Label the x-axis with month names and the y-axis with "Temperature (°C)".
    * Add tooltips to display the month and relevant statistical values (min, max, media

In [ ]:
import numpy as np
from bokeh.models import Whisker
from bokeh.transform import factor_cmap

# Extraction du mois
df['Month'] = df['Date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df['MonthName'] = df['Month'].map(lambda m: month_names[m-1])

# Stats par mois
groups = df.groupby('Month')['Temperature']
q1 = groups.quantile(0.25)
q2 = groups.quantile(0.5)
q3 = groups.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
qmin = groups.min()
qmax = groups.max()

upper = [min(u, m) for u, m in zip(upper, qmax)]
lower = [max(l, m) for l, m in zip(lower, qmin)]

months = [month_names[m-1] for m in range(1,13)]

source3 = ColumnDataSource(data=dict(
    months=months,
    q1=q1.values, q2=q2.values, q3=q3.values,
    upper=upper, lower=lower
))

p3 = figure(x_range=months, title="Distribution mensuelle des températures",
            width=900, height=400, y_axis_label="Temperature (°C)",
            tools="pan,wheel_zoom,reset")

# Whiskers
whisker_upper = Whisker(base="months", upper="upper", lower="q3", source=source3,
                        line_color="black")
whisker_lower = Whisker(base="months", upper="q1", lower="lower", source=source3,
                        line_color="black")
p3.add_layout(whisker_upper)
p3.add_layout(whisker_lower)

# Boxes
p3.vbar("months", 0.6, "q2", "q3", source=source3, fill_color="steelblue",
        line_color="black", fill_alpha=0.7)
p3.vbar("months", 0.6, "q1", "q2", source=source3, fill_color="lightblue",
        line_color="black", fill_alpha=0.7)

# Mediane
p3.segment("months", "q2", "months", "q2", source=source3, line_color="red",
           line_width=2)

hover3 = HoverTool(tooltips=[("Mois", "@months"),
                              ("Mediane", "@q2{0.1} °C"),
                              ("Q1", "@q1{0.1} °C"),
                              ("Q3", "@q3{0.1} °C")])
p3.add_tools(hover3)

show(p3)

In [5]:
from bokeh.plotting import output_notebook, show
import pandas as pd

output_notebook()  # Enable Bokeh output in Jupyter Notebook

# Load the Dataset
df = pd.read_csv("./datasets/daily-minimum-temperatures-in-melbourne.csv")

# Now you can proceed with your Bokeh plotting code!
print(df.head()) # Just to see if the dataframe loaded correctly


# Rename columns for clarity
df.columns = ['Date', 'Temperature']

# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Remove '?' from the 'Temperature' column and convert to numeric
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'])

df

Loading BokehJS ...

         Date DailyTemperature
0  1981-01-01             20.7
1  1981-01-02             17.9
2  1981-01-03             18.8
3  1981-01-04             14.6
4  1981-01-05             15.8


,Date,Temperature
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8
...,...,...
3645,1990-12-27,14.0
3646,1990-12-28,13.6
3647,1990-12-29,13.5
3648,1990-12-30,15.7


4.  Create box plots to visualize the distribution of temperatures for each year,
    and use color mapping to highlight temperature variations.

    * Extract the year from the 'Date' column and create a new 'Year' column.
    * Group the data by 'Year' and prepare it for plotting.
    * Use Bokeh's box plot elements to visualize the distribution for each year.
    * Label the x-axis with the 'Year' and the y-axis with "Temperature (°C)".
    * Use `factor_cmap` to color the boxes based on the median temperature of each year.
    * Add tooltips to display the year and relevant statistical values (min, max, median, etc.).
    * Enable pan, wheel zoom, and reset tools.

In [ ]:
# Box plots par annee
df['Year'] = df['Date'].dt.year.astype(str)

years = sorted(df['Year'].unique().tolist())
groups_y = df.groupby('Year')['Temperature']
q1y = groups_y.quantile(0.25)
q2y = groups_y.quantile(0.5)
q3y = groups_y.quantile(0.75)
iqry = q3y - q1y
uppery = q3y + 1.5 * iqry
lowery = q1y - 1.5 * iqry
qminy = groups_y.min()
qmaxy = groups_y.max()

uppery = [min(u, m) for u, m in zip(uppery, qmaxy)]
lowery = [max(l, m) for l, m in zip(lowery, qminy)]

source4 = ColumnDataSource(data=dict(
    years=years,
    q1=q1y.values, q2=q2y.values, q3=q3y.values,
    upper=uppery, lower=lowery
))

colors = ["#2b83ba","#abdda4","#ffffbf","#fdae61","#d7191c",
          "#2b83ba","#abdda4","#ffffbf","#fdae61","#d7191c"]

p4 = figure(x_range=years, title="Distribution annuelle des températures",
            width=900, height=400, y_axis_label="Temperature (°C)",
            tools="pan,wheel_zoom,reset")

whisker_u4 = Whisker(base="years", upper="upper", lower="q3", source=source4, line_color="black")
whisker_l4 = Whisker(base="years", upper="q1", lower="lower", source=source4, line_color="black")
p4.add_layout(whisker_u4)
p4.add_layout(whisker_l4)

p4.vbar("years", 0.6, "q2", "q3", source=source4,
        fill_color=factor_cmap('years', palette=colors[:len(years)], factors=years),
        line_color="black", fill_alpha=0.8)
p4.vbar("years", 0.6, "q1", "q2", source=source4,
        fill_color=factor_cmap('years', palette=colors[:len(years)], factors=years),
        line_color="black", fill_alpha=0.6)

hover4 = HoverTool(tooltips=[("Année", "@years"),
                              ("Médiane", "@q2{0.1} °C"),
                              ("Q1", "@q1{0.1} °C"),
                              ("Q3", "@q3{0.1} °C")])
p4.add_tools(hover4)

show(p4)

Question 5: Interactive Time Range Selection

5.  Create an interactive line plot where the user can select a specific time range
    to view using a date range slider.

    * Create a basic line plot of 'Temperature' over 'Date'.
    * Implement a date range slider using Bokeh widgets to allow users to select a start and end date.
    * Update the plot dynamically based on the selected date range.
    * Add tooltips to display the date and temperature.
    * Enable pan, wheel zoom, and reset tools.

In [ ]:
from bokeh.models import DateRangeSlider, CustomJS
from bokeh.layouts import column

# Donnees
source5_full = ColumnDataSource(df[['Date','Temperature']])
source5 = ColumnDataSource(df[['Date','Temperature']])

p5 = figure(title="Températures avec sélection de plage", x_axis_type="datetime",
            width=900, height=400,
            x_axis_label="Date", y_axis_label="Temperature (°C)",
            tools="pan,wheel_zoom,reset")

p5.line('Date', 'Temperature', source=source5, line_width=1, color="steelblue")

hover5 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Temp", "@Temperature{0.1} °C")],
                   formatters={"@Date": "datetime"})
p5.add_tools(hover5)

date_min = df['Date'].min()
date_max = df['Date'].max()

slider = DateRangeSlider(title="Plage de dates", start=date_min, end=date_max,
                         value=(date_min, date_max), step=1, width=860)

callback = CustomJS(args=dict(source=source5, full=source5_full), code="""
    var data = full.data;
    var start = cb_obj.value[0];
    var end = cb_obj.value[1];
    var new_data = {'Date': [], 'Temperature': []};
    for (var i = 0; i < data['Date'].length; i++) {
        if (data['Date'][i] >= start && data['Date'][i] <= end) {
            new_data['Date'].push(data['Date'][i]);
            new_data['Temperature'].push(data['Temperature'][i]);
        }
    }
    source.data = new_data;
    source.change.emit();
""")

slider.js_on_change('value', callback)

show(column(slider, p5))

Question 6: Time Series Decomposition Visualization

6.  Perform a simple time series decomposition to visualize the trend and seasonality
    components of the temperature data.

    * Resample the data to monthly frequency and calculate the monthly average temperature.
    * Use a simple moving average to estimate the trend component.
    * Calculate the seasonal component by subtracting the trend from the original monthly data.
    * Create three separate Bokeh plots: one for the original monthly data, one for the trend,
        and one for the seasonal component.
    * Ensure the plots are aligned and share the same x-axis (Date).
    * Add tooltips to each plot to display the date and corresponding value.
    * Enable pan, wheel zoom, and reset tools for each plot.

In [ ]:
from bokeh.layouts import column as bk_column

# Resample mensuel
df_monthly = df.set_index('Date').resample('M')['Temperature'].mean().reset_index()
df_monthly.columns = ['Date', 'Temp_Mensuelle']

# Trend : moyenne glissante 12 mois
df_monthly['Trend'] = df_monthly['Temp_Mensuelle'].rolling(window=12, center=True).mean()

# Composante saisonniere
df_monthly['Seasonal'] = df_monthly['Temp_Mensuelle'] - df_monthly['Trend']

src_m = ColumnDataSource(df_monthly)

# Plot 1 : donnees mensuelles
p6a = figure(title="Température mensuelle moyenne", x_axis_type="datetime",
             width=900, height=250, x_axis_label="Date", y_axis_label="Temp (°C)",
             tools="pan,wheel_zoom,reset")
p6a.line('Date', 'Temp_Mensuelle', source=src_m, color="steelblue", line_width=1.5)
p6a.add_tools(HoverTool(tooltips=[("Date","@Date{%F}"),("Temp","@Temp_Mensuelle{0.1}")],
                         formatters={"@Date":"datetime"}))

# Plot 2 : trend
p6b = figure(title="Composante Tendance (MA 12 mois)", x_axis_type="datetime",
             width=900, height=250, x_range=p6a.x_range,
             x_axis_label="Date", y_axis_label="Trend (°C)",
             tools="pan,wheel_zoom,reset")
p6b.line('Date', 'Trend', source=src_m, color="tomato", line_width=2)
p6b.add_tools(HoverTool(tooltips=[("Date","@Date{%F}"),("Trend","@Trend{0.1}")],
                         formatters={"@Date":"datetime"}))

# Plot 3 : saisonnalite
p6c = figure(title="Composante Saisonnière", x_axis_type="datetime",
             width=900, height=250, x_range=p6a.x_range,
             x_axis_label="Date", y_axis_label="Seasonal (°C)",
             tools="pan,wheel_zoom,reset")
p6c.line('Date', 'Seasonal', source=src_m, color="seagreen", line_width=1.5)
p6c.add_tools(HoverTool(tooltips=[("Date","@Date{%F}"),("Seasonal","@Seasonal{0.1}")],
                         formatters={"@Date":"datetime"}))

show(bk_column(p6a, p6b, p6c))